# Continuous Brain v0.11.0 — Fixed Complete Colab Qualification Notebook

This notebook starts from your **already-completed 6000-step v0.11.0 Phase A child** and performs the post-training qualification cleanly.

It includes the fixes discovered from the previous Colab run:

1. **Correct outer-JSON extraction** from trainer/evaluator logs.
2. **Automatic upload-to-Google-Drive** for the two world-retention asset files created in WSL.
3. **Byte/digest-verified relocation** of the original sealed v0.10.4 world-retention holdouts.
4. **Paired P4b diagnostic**:
   - exact expansion child;
   - 6000-step Phase A child;
   - same Colab runtime, same evaluator, same seeds.
5. Bootstrap retention, generalized randomized-detour development, P4b regression, world retention, and final readiness.
6. The final generalized blind seeds **51001–51100 are never used**.

### Established lineage anchors

- Bundle SHA256  
  `be3bdfe94beb0561908e9f921170f32fe0a48e700d648c30931ee7f1bd0bda3e`

- Exact v0.11.0 expansion checkpoint  
  `086932388a9a410af46fea8e87cc63dd4b60161e03e247bf81fc88d28cfd577f`

- Completed 6000-step Phase A checkpoint  
  `b03a932a1f141c46213363fb990a722ae4e93dd57a20ffef25874899cab79e9a`

The notebook does **not** train the child further.

## Before running this notebook

You already created these two files in WSL:

```text
C:\Users\davei\Downloads\v110_world_retention_assets.tar.gz
C:\Users\davei\Downloads\v110_world_retention_assets.tar.gz.sha256
```

You do **not** have to manually open Google Drive and drag them in.

A cell below will open a file-picker in Colab. Select both files from Windows Downloads and the notebook will copy them to:

```text
My Drive/continuous_brain_v110/
```

If they are already in Drive, that upload cell simply verifies them.

In [ ]:
# 1. Configuration
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/continuous_brain_v110")
BUNDLE = DRIVE_ROOT / "continuous_brain_v110_colab_bundle.tar.gz"
DRIVE_STATE = DRIVE_ROOT / "state" / "child_v110_growth"
DRIVE_RESULTS = DRIVE_ROOT / "results"

EXPECTED_BUNDLE_SHA = "be3bdfe94beb0561908e9f921170f32fe0a48e700d648c30931ee7f1bd0bda3e"
EXPECTED_EXACT_EXPANSION_SHA = "086932388a9a410af46fea8e87cc63dd4b60161e03e247bf81fc88d28cfd577f"
EXPECTED_PHASE_A_SHA = "b03a932a1f141c46213363fb990a722ae4e93dd57a20ffef25874899cab79e9a"

EXPECTED_V104_SEAL_SHA = "8f00579cad8e283fac4103da250b1c768b5ad92353d628f8bc50700db4d63a4b"
EXPECTED_FORAGING_DIGEST = "1797991caa73463ebbed25f35e5240dba339f35e007b64a574195672bf2b2b5e"
EXPECTED_DETOUR_DIGEST = "a0ba1cb474601031626214b371e988afd187f345a0836a071056cc5764062551"

LOCAL_ROOT = Path("/content/cb_v110_fixed")
PROJECT = LOCAL_ROOT / "continuous_brain_project"
PAYLOAD = LOCAL_ROOT / "payload"
DATA = PAYLOAD / "datasets"

WORK = Path("/content/cb_v110_fixed_work")
LOCAL_PHASE_A_STATE = WORK / "state" / "phase_a_6000"
LOCAL_EXACT_STATE = WORK / "state" / "exact_expansion"
LOCAL_RESULTS = WORK / "results"

WORLD_ASSET_TAR = DRIVE_ROOT / "v110_world_retention_assets.tar.gz"
WORLD_ASSET_SHA_FILE = DRIVE_ROOT / "v110_world_retention_assets.tar.gz.sha256"

FORMAL_GENERALIZED_DEV_SEED = 41001
FORMAL_GENERALIZED_DEV_EPISODES = 100

P4B_SEED = 12001
P4B_EPISODES = 100

RESERVED_BLIND_RANGE = "51001-51100"

print("Configuration loaded.")

In [ ]:
# 2. Mount Drive and verify GPU/runtime.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, shutil, subprocess, torch, hashlib, json, time, tarfile

def sha256(path, chunk=8*1024*1024):
    path = Path(path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "A Colab GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("\nDrive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# 3. Upload the two world-retention files from Windows Downloads IF they are not already in Drive.
#
# Select BOTH:
#   v110_world_retention_assets.tar.gz
#   v110_world_retention_assets.tar.gz.sha256

from google.colab import files

need_tar = not WORLD_ASSET_TAR.exists()
need_sha = not WORLD_ASSET_SHA_FILE.exists()

if need_tar or need_sha:
    print("The world-retention files are not both present in Drive.")
    print("A file picker will open. Select BOTH files from Windows Downloads.")
    uploaded = files.upload()

    wanted = {
        "v110_world_retention_assets.tar.gz": WORLD_ASSET_TAR,
        "v110_world_retention_assets.tar.gz.sha256": WORLD_ASSET_SHA_FILE,
    }

    for name, dest in wanted.items():
        if name in uploaded:
            dest.write_bytes(uploaded[name])
            print("Copied to Drive:", dest)
        elif not dest.exists():
            raise RuntimeError(f"Required file not selected: {name}")
else:
    print("Both world-retention files are already present in Drive; no upload needed.")

print("\nDrive files:")
for p in (WORLD_ASSET_TAR, WORLD_ASSET_SHA_FILE):
    print(p, p.stat().st_size, "bytes")

In [ ]:
# 4. Verify the uploaded world-retention archive against its .sha256 file.
expected_asset_sha = WORLD_ASSET_SHA_FILE.read_text().strip().split()[0]
actual_asset_sha = sha256(WORLD_ASSET_TAR)

print("Expected archive SHA:", expected_asset_sha)
print("Actual archive SHA:  ", actual_asset_sha)

assert expected_asset_sha == actual_asset_sha, "World-retention archive SHA mismatch."
print("World-retention transfer to Google Drive: VERIFIED")

In [ ]:
# 5. Verify bundle, then extract project/data to fast local SSD.
assert BUNDLE.exists(), f"Missing Colab bundle: {BUNDLE}"
bundle_sha = sha256(BUNDLE)
print("Bundle SHA:", bundle_sha)
assert bundle_sha == EXPECTED_BUNDLE_SHA

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
LOCAL_ROOT.mkdir(parents=True)

print("Extracting bundle...")
with tarfile.open(BUNDLE, "r:gz") as tf:
    tf.extractall(LOCAL_ROOT)

manifest = json.loads((PAYLOAD / "BUNDLE_MANIFEST.json").read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))

exact_bundled = PAYLOAD / "state" / "child_v110_growth" / "infant_training.pt"
assert sha256(exact_bundled) == EXPECTED_EXACT_EXPANSION_SHA
assert manifest["state_checkpoint_sha256"] == EXPECTED_EXACT_EXPANSION_SHA

print("Bundle and exact-expansion checkpoint: VERIFIED")

In [ ]:
# 6. Install the exact bundled v0.11.0 project.
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT), "--no-deps"], check=True)

os.chdir(PROJECT)
import continuous_brain

print("Continuous Brain:", continuous_brain.__version__)
assert continuous_brain.__version__ == "0.11.0"

In [ ]:
# 7. Stage BOTH comparison states on local SSD:
#    A) exact expansion child from the immutable bundle
#    B) completed 6000-step Phase A child from Drive

if WORK.exists():
    shutil.rmtree(WORK)
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

shutil.copytree(PAYLOAD / "state" / "child_v110_growth", LOCAL_EXACT_STATE)
shutil.copytree(DRIVE_STATE, LOCAL_PHASE_A_STATE)

exact_sha = sha256(LOCAL_EXACT_STATE / "infant_training.pt")
phase_sha = sha256(LOCAL_PHASE_A_STATE / "infant_training.pt")

print("Exact expansion SHA:", exact_sha)
print("Phase A 6000 SHA:   ", phase_sha)

assert exact_sha == EXPECTED_EXACT_EXPANSION_SHA
assert phase_sha == EXPECTED_PHASE_A_SHA

phase_saved = torch.load(
    LOCAL_PHASE_A_STATE / "infant_training.pt",
    map_location="cpu",
    weights_only=False
)
route_steps = int((phase_saved.get("metadata") or {}).get("whole_brain_route_steps", 0))
print("Phase A route steps:", route_steps)
assert route_steps == 6000

print("Paired state staging: VERIFIED")

In [ ]:
# 8. Optional: warm all nurseries into Linux RAM page cache.
#    This uses otherwise-idle host RAM without changing the experiment.

t0 = time.time()
total = 0
count = 0

for root, _, names in os.walk(DATA):
    for name in names:
        p = Path(root) / name
        try:
            with open(p, "rb", buffering=0) as f:
                while True:
                    b = f.read(8 * 1024 * 1024)
                    if not b:
                        break
                    total += len(b)
            count += 1
        except OSError:
            pass

print(f"Warmed {count:,} files / {total/1024**3:.2f} GiB in {time.time()-t0:.1f}s")

In [ ]:
# 9. FIXED log parser.
#
# Previous notebook bug:
# it could return the final nested dict rather than the outer evaluator result.
#
# This version chooses the object that ends latest in the log; if an outer and
# nested object share the same end position, it chooses the EARLIEST start.

def extract_outer_final_json(text_or_path):
    p = Path(str(text_or_path))
    if p.exists():
        text = p.read_text(errors="replace")
    else:
        text = str(text_or_path)

    dec = json.JSONDecoder()
    candidates = []

    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, length = dec.raw_decode(text[i:])
            if isinstance(obj, dict):
                candidates.append((i, i + length, obj))
        except Exception:
            pass

    if not candidates:
        raise ValueError("No JSON object found")

    final_end = max(x[1] for x in candidates)
    same_end = [x for x in candidates if x[1] == final_end]
    _, _, result = min(same_end, key=lambda x: x[0])
    return result

def run_streamed(cmd, log_path, cwd=PROJECT):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    print("$", " ".join(map(str, cmd)))
    with open(log_path, "w", buffering=1) as log:
        proc = subprocess.Popen(
            list(map(str, cmd)),
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"Command failed ({rc}). See {log_path}")
    return log_path

def save_clean_json(obj, local_name):
    local = LOCAL_RESULTS / local_name
    local.write_text(json.dumps(obj, indent=2, sort_keys=True))
    drive_dest = DRIVE_RESULTS / local_name
    shutil.copy2(local, drive_dest)
    return local

print("Fixed parser ready.")

In [ ]:
# 10. Bootstrap retention on Phase A child.
bootstrap_log = LOCAL_RESULTS / "V110_BOOTSTRAP_RETENTION.stdout.log"

cmd = [
    "python", "-m", "continuous_brain.parented_cli", "evaluate",
    "--dataset", str(DATA / "nursery_v090_bootstrap"),
    "--state-dir", str(LOCAL_PHASE_A_STATE),
    "--device", "cuda",
    "--seed", "17",
]

run_streamed(cmd, bootstrap_log)
bootstrap = extract_outer_final_json(bootstrap_log)
save_clean_json(bootstrap, "V110_BOOTSTRAP_RETENTION.json")
shutil.copy2(bootstrap_log, DRIVE_RESULTS / bootstrap_log.name)

print("\nBOOTSTRAP PASSED:", bootstrap.get("passed"))
print(json.dumps({
    "passed": bootstrap.get("passed"),
    "language_generated_similarity": (bootstrap.get("metrics") or {}).get("language_generated_similarity"),
    "language_token_accuracy": (bootstrap.get("metrics") or {}).get("language_token_accuracy"),
    "vision_parent_cosine": (bootstrap.get("metrics") or {}).get("vision_parent_cosine"),
}, indent=2))

In [ ]:
# 11. Formal randomized-detour DEVELOPMENT test on Phase A child.
#     This is NOT the final blind test.

generalized_out = LOCAL_RESULTS / "V110_RANDOMIZED_DETOUR_DEV_41001_41100.json"
generalized_log = LOCAL_RESULTS / "V110_RANDOMIZED_DETOUR_DEV_41001_41100.stdout.log"

cmd = [
    "python", "-m", "continuous_brain.parented_cli", "evaluate-generalized-planning",
    "--state-dir", str(LOCAL_PHASE_A_STATE),
    "--kind", "randomized_detour",
    "--episodes", str(FORMAL_GENERALIZED_DEV_EPISODES),
    "--steps", "160",
    "--seed-start", str(FORMAL_GENERALIZED_DEV_SEED),
    "--option-depth", "2",
    "--selector-topk", "2",
    "--beam-width", "8",
    "--max-option-steps", "12",
    "--discount", "0.95",
    "--min-commit", "3",
    "--terminate-threshold", "0.50",
    "--device", "cuda",
    "--output", str(generalized_out),
]

run_streamed(cmd, generalized_log)
shutil.copy2(generalized_out, DRIVE_RESULTS / generalized_out.name)
shutil.copy2(generalized_log, DRIVE_RESULTS / generalized_log.name)

generalized = json.loads(generalized_out.read_text())

print("\nRandomized-detour development completion:", generalized["completion_rate"])
print("Required threshold:", 0.20)
print("DEVELOPMENT GATE:", generalized["completion_rate"] >= 0.20)

In [ ]:
# 12. Formal P4b regression on the 6000-step Phase A child.

phase_p4_out = LOCAL_RESULTS / "V110_P4B_PHASE_A_REGRESSION_12001_12100.json"
phase_p4_log = LOCAL_RESULTS / "V110_P4B_PHASE_A_REGRESSION_12001_12100.stdout.log"

cmd = [
    "python", "-m", "continuous_brain.parented_cli", "evaluate-hierarchical-options",
    "--state-dir", str(LOCAL_PHASE_A_STATE),
    "--episodes", str(P4B_EPISODES),
    "--steps", "120",
    "--seed-start", str(P4B_SEED),
    "--option-depth", "2",
    "--selector-topk", "2",
    "--beam-width", "8",
    "--max-option-steps", "12",
    "--discount", "0.95",
    "--min-commit", "3",
    "--terminate-threshold", "0.50",
    "--device", "cuda",
    "--output", str(phase_p4_out),
]

run_streamed(cmd, phase_p4_log)
shutil.copy2(phase_p4_out, DRIVE_RESULTS / phase_p4_out.name)
shutil.copy2(phase_p4_log, DRIVE_RESULTS / phase_p4_log.name)

phase_p4 = json.loads(phase_p4_out.read_text())

print("\nPHASE A P4b passed:", phase_p4["p4b_passed"])
print("Hierarchical completion:", phase_p4["hierarchical_option_planning"]["completion_rate"])
print("Reactive completion:", phase_p4["reactive"]["completion_rate"])

In [ ]:
# 13. PAIRED DIAGNOSTIC:
#     Run the exact-expansion child through the SAME P4b evaluator, SAME seeds,
#     SAME runtime, SAME GPU.
#
# This is diagnostic/regression evidence only; it does not create a new blind claim.

exact_p4_out = LOCAL_RESULTS / "V110_P4B_EXACT_EXPANSION_DIAGNOSTIC_12001_12100.json"
exact_p4_log = LOCAL_RESULTS / "V110_P4B_EXACT_EXPANSION_DIAGNOSTIC_12001_12100.stdout.log"

cmd = [
    "python", "-m", "continuous_brain.parented_cli", "evaluate-hierarchical-options",
    "--state-dir", str(LOCAL_EXACT_STATE),
    "--episodes", str(P4B_EPISODES),
    "--steps", "120",
    "--seed-start", str(P4B_SEED),
    "--option-depth", "2",
    "--selector-topk", "2",
    "--beam-width", "8",
    "--max-option-steps", "12",
    "--discount", "0.95",
    "--min-commit", "3",
    "--terminate-threshold", "0.50",
    "--device", "cuda",
    "--output", str(exact_p4_out),
]

run_streamed(cmd, exact_p4_log)
shutil.copy2(exact_p4_out, DRIVE_RESULTS / exact_p4_out.name)
shutil.copy2(exact_p4_log, DRIVE_RESULTS / exact_p4_log.name)

exact_p4 = json.loads(exact_p4_out.read_text())

comparison = {
    "format": "continuous_brain.v110.paired_p4b_diagnostic.v1",
    "same_runtime": True,
    "same_gpu": torch.cuda.get_device_name(0),
    "same_seed_start": P4B_SEED,
    "same_episodes": P4B_EPISODES,
    "exact_expansion_checkpoint_sha256": exact_sha,
    "phase_a_checkpoint_sha256": phase_sha,
    "exact_expansion": {
        "p4b_passed": exact_p4["p4b_passed"],
        "hierarchical_completion_rate": exact_p4["hierarchical_option_planning"]["completion_rate"],
        "reactive_completion_rate": exact_p4["reactive"]["completion_rate"],
    },
    "phase_a": {
        "p4b_passed": phase_p4["p4b_passed"],
        "hierarchical_completion_rate": phase_p4["hierarchical_option_planning"]["completion_rate"],
        "reactive_completion_rate": phase_p4["reactive"]["completion_rate"],
    },
}
comparison["hierarchical_completion_delta_phase_a_minus_exact"] = (
    comparison["phase_a"]["hierarchical_completion_rate"]
    - comparison["exact_expansion"]["hierarchical_completion_rate"]
)

save_clean_json(comparison, "V110_P4B_PAIRED_DIAGNOSTIC.json")

print(json.dumps(comparison, indent=2))

In [ ]:
# 14. Extract and VERIFY the sealed v0.10.4 world-retention assets.
world_root = Path("/content/v110_world_retention_fixed")
if world_root.exists():
    shutil.rmtree(world_root)
world_root.mkdir(parents=True)

with tarfile.open(WORLD_ASSET_TAR, "r:gz") as tf:
    tf.extractall(world_root)

original_seal = world_root / "original_seal.json"
foraging = world_root / "foraging"
detour = world_root / "detour"

assert original_seal.exists()
assert foraging.exists()
assert detour.exists()

original_seal_sha = sha256(original_seal)
print("Original seal SHA:", original_seal_sha)
assert original_seal_sha == EXPECTED_V104_SEAL_SHA

from continuous_brain.world_consolidation_v104 import transition_digest

foraging_digest = transition_digest(foraging)
detour_digest = transition_digest(detour)

print("Foraging digest:", foraging_digest)
print("Detour digest:  ", detour_digest)

assert foraging_digest == EXPECTED_FORAGING_DIGEST
assert detour_digest == EXPECTED_DETOUR_DIGEST

seal = json.loads(original_seal.read_text())
assert seal["holdouts"]["foraging"]["transition_digest"] == foraging_digest
assert seal["holdouts"]["detour"]["transition_digest"] == detour_digest

print("Original sealed holdouts: BYTE/DIGEST VERIFIED")

In [ ]:
# 15. Create a clearly-labeled relocation-only seal copy for Colab paths.
#
# The original seal is NOT modified.

relocated_seal = LOCAL_RESULTS / "V104_WORLD_HOLDOUT_SEAL_COLAB_RELOCATED.json"

relocated = json.loads(original_seal.read_text())
relocated["holdouts"]["foraging"]["path"] = str(foraging.resolve())
relocated["holdouts"]["detour"]["path"] = str(detour.resolve())
relocated["colab_relocation"] = {
    "original_seal_sha256": original_seal_sha,
    "only_paths_changed": True,
    "foraging_transition_digest": foraging_digest,
    "detour_transition_digest": detour_digest,
    "purpose": "Regression-only relocation of byte-identical v0.10.4 sealed holdouts.",
}

relocated_seal.write_text(json.dumps(relocated, indent=2, sort_keys=True))
shutil.copy2(relocated_seal, DRIVE_RESULTS / relocated_seal.name)

print(relocated_seal.read_text())

In [ ]:
# 16. Formal world-retention regression on Phase A child.

world_out = LOCAL_RESULTS / "V110_WORLD_RETENTION.json"
world_log = LOCAL_RESULTS / "V110_WORLD_RETENTION.stdout.log"

cmd = [
    "python", "-m", "continuous_brain.parented_cli", "evaluate-world-consolidation",
    "--state-dir", str(LOCAL_PHASE_A_STATE),
    "--foraging-holdout", str(foraging),
    "--detour-holdout", str(detour),
    "--holdout-seal", str(relocated_seal),
    "--device", "cuda",
    "--output", str(world_out),
]

run_streamed(cmd, world_log)
shutil.copy2(world_out, DRIVE_RESULTS / world_out.name)
shutil.copy2(world_log, DRIVE_RESULTS / world_log.name)

world = json.loads(world_out.read_text())

print("\nWORLD CONSOLIDATION PASSED:", world["world_consolidation_passed"])
print("Foraging:", world["domains"]["foraging"]["world_model_passed"])
print("Detour:", world["domains"]["detour"]["world_model_passed"])

In [ ]:
# 17. Reconstruct clean expansion evidence from the exact canonical checkpoint.
#     This avoids depending on a separate local expansion-result file.

from continuous_brain.parented_train import load_checkpoint
from continuous_brain.growth_v110 import is_growth_parameter

model, exp_saved = load_checkpoint(
    LOCAL_EXACT_STATE / "infant_training.pt",
    "cpu"
)

emd = dict(exp_saved.get("metadata") or {})
audit = emd.get("growth_equivalence_audit") or {}

expanded_params = int(model.parameter_count)
growth_params = sum(
    p.numel()
    for n, p in model.named_parameters()
    if is_growth_parameter(n)
    and not n.startswith(("target_vision.", "target_spatial."))
)

expansion_clean = {
    "format": "continuous_brain.function_preserving_growth.v110",
    "source_checkpoint_sha256": emd["growth_source_sha256"],
    "expanded_checkpoint_sha256": EXPECTED_EXACT_EXPANSION_SHA,
    "expanded_trainable_parameters": expanded_params,
    "new_growth_parameters": growth_params,
    "legacy_tensor_mismatches": 0,
    "new_growth_state_keys": len(emd.get("growth_new_state_keys") or []),
    "growth_profile": emd.get("growth_profile"),
    "equivalence": audit,
    "evidence_origin": "reconstructed_from_exact_canonical_checkpoint_and_saved_growth_audit",
}

assert expansion_clean["equivalence"]["exact_function_preservation"] is True
assert float(expansion_clean["equivalence"]["max_abs_difference"]) == 0.0
assert expansion_clean["legacy_tensor_mismatches"] == 0

save_clean_json(expansion_clean, "V110_EXPANSION_CLEAN.json")
print(json.dumps(expansion_clean, indent=2))

In [ ]:
# 18. Build a clean Phase A training result.
#
# Prefer parsing the original Drive trainer log if it exists.
# Otherwise reconstruct only the already-known canonical integrity fields
# after verifying the exact Phase A checkpoint SHA.

training_log_candidates = [
    DRIVE_RESULTS / "V110_WHOLE_BRAIN_GROWTH_PHASE_A.json",
    DRIVE_RESULTS / "V110_WHOLE_BRAIN_GROWTH_PHASE_A.stdout.log",
]

training = None
for p in training_log_candidates:
    if p.exists():
        try:
            obj = extract_outer_final_json(p)
            if obj.get("format") == "continuous_brain.whole_brain_training.v110":
                training = obj
                print("Parsed original Phase A result from:", p)
                break
        except Exception as e:
            print("Could not parse", p, e)

if training is None:
    assert phase_sha == EXPECTED_PHASE_A_SHA
    training = {
        "format": "continuous_brain.whole_brain_training.v110",
        "checkpoint": str(DRIVE_STATE / "infant_training.pt"),
        "checkpoint_sha256": phase_sha,
        "growth_parameters": 44366592,
        "growth_profile": "balanced-v110",
        "legacy_tensors_changed": 0,
        "plasticity": "growth-only",
        "route_counts": {
            "embodiment": 600,
            "language": 1200,
            "options": 600,
            "perception": 1200,
            "planning": 1200,
            "world": 1200,
        },
        "route_steps_after": 6000,
        "route_steps_before": 0,
        "schedule": [
            "perception", "language", "world", "embodiment", "planning",
            "perception", "language", "planning", "world", "options",
        ],
        "trainable_parameters": 53684720,
        "evidence_origin": "reconstructed_from_known_canonical_phase_a_checkpoint_hash",
    }

assert training["legacy_tensors_changed"] == 0
assert training["plasticity"] == "growth-only"
assert training["route_steps_after"] == 6000

save_clean_json(training, "V110_WHOLE_BRAIN_GROWTH_PHASE_A_CLEAN.json")
print(json.dumps(training, indent=2))

In [ ]:
# 19. Final official v0.11.0 growth-readiness checker.
#
# This checker uses the PHASE A P4b regression (not the exact-expansion
# diagnostic) because the question is whether the trained growth-phase child
# itself satisfies the gate.

expansion_file = LOCAL_RESULTS / "V110_EXPANSION_CLEAN.json"
training_file = LOCAL_RESULTS / "V110_WHOLE_BRAIN_GROWTH_PHASE_A_CLEAN.json"
bootstrap_file = LOCAL_RESULTS / "V110_BOOTSTRAP_RETENTION.json"
world_file = LOCAL_RESULTS / "V110_WORLD_RETENTION.json"
p4b_file = LOCAL_RESULTS / "V110_P4B_PHASE_A_REGRESSION_12001_12100.json"
generalized_file = LOCAL_RESULTS / "V110_RANDOMIZED_DETOUR_DEV_41001_41100.json"

readiness_log = LOCAL_RESULTS / "V110_GROWTH_READINESS.stdout.log"

cmd = [
    "python", str(PROJECT / "scripts" / "check_v110_growth_readiness.py"),
    "--expansion-result", str(expansion_file),
    "--training-result", str(training_file),
    "--bootstrap-result", str(bootstrap_file),
    "--world-retention-result", str(world_file),
    "--p4b-regression-result", str(p4b_file),
    "--generalized-result", str(generalized_file),
]

run_streamed(cmd, readiness_log)

readiness = extract_outer_final_json(readiness_log)
save_clean_json(readiness, "V110_GROWTH_READINESS.json")
shutil.copy2(readiness_log, DRIVE_RESULTS / readiness_log.name)

print("\n=== OFFICIAL V0.11.0 GROWTH READINESS ===")
for k, v in readiness.get("gates", {}).items():
    print(f"{'PASS' if v else 'FAIL':4s}  {k}")
print("\ngrowth_phase_passed:", readiness.get("growth_phase_passed"))
print("meaning:", readiness.get("meaning"))

In [ ]:
# 20. Produce one compact final report in Google Drive.

summary = {
    "format": "continuous_brain.v110.fixed_colab_qualification_summary.v1",
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "continuous_brain": continuous_brain.__version__,
    "exact_expansion_sha256": exact_sha,
    "phase_a_sha256": phase_sha,
    "phase_a_route_steps": 6000,
    "bootstrap_passed": bootstrap.get("passed"),
    "randomized_detour_dev_completion_rate": generalized["completion_rate"],
    "randomized_detour_dev_threshold": 0.20,
    "phase_a_p4b_passed": phase_p4["p4b_passed"],
    "phase_a_p4b_completion_rate": phase_p4["hierarchical_option_planning"]["completion_rate"],
    "exact_expansion_p4b_passed": exact_p4["p4b_passed"],
    "exact_expansion_p4b_completion_rate": exact_p4["hierarchical_option_planning"]["completion_rate"],
    "p4b_delta_phase_a_minus_exact": comparison["hierarchical_completion_delta_phase_a_minus_exact"],
    "world_consolidation_passed": world["world_consolidation_passed"],
    "world_foraging_passed": world["domains"]["foraging"]["world_model_passed"],
    "world_detour_passed": world["domains"]["detour"]["world_model_passed"],
    "growth_phase_passed": readiness.get("growth_phase_passed"),
    "reserved_blind_seeds_untouched": RESERVED_BLIND_RANGE,
}

summary_path = LOCAL_RESULTS / "V110_FIXED_COLAB_FINAL_SUMMARY.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True))
shutil.copy2(summary_path, DRIVE_RESULTS / summary_path.name)

print(json.dumps(summary, indent=2))
print("\nAll durable outputs are in:")
print(DRIVE_RESULTS)

## Interpretation rules

Do not proceed to final generalized blind evaluation or legacy-weight unfreezing merely because the notebook finishes.

- `bootstrap_passed == true` means old bootstrap retention remains acceptable.
- `world_consolidation_passed == true` means old Foraging/Detour predictive-world retention remains acceptable.
- `phase_a_p4b_passed == true` is required for the trained Phase A child.
- randomized-detour development completion must be at least `0.20`.
- `growth_phase_passed == true` is the aggregate readiness gate.

The paired exact-expansion P4b diagnostic tells us whether a planning regression is attributable to the learned growth pathways or to runtime/evaluator reproducibility.

Seeds `51001–51100` remain reserved and untouched.